# Notebook 01 — Skill Bayesian Networks + Markov Chain
**InterviewIQ · Unit I (Bayesian Networks: causal / evidential reasoning, independencies) and Unit IV (Markov chains).**

One Bayesian Network per role. Skill nodes come from the real categories in the question bank. Edges (prerequisites) and CPD numbers are **hand-set priors**, not learned.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))   # so notebooks can import the project modules
DATA = '../data'
import pandas as pd
from skill_network import *
qb = pd.read_csv(f'{DATA}/question_bank.csv')
print(qb.role.value_counts())
print('Category->skill coverage problems:', check_category_coverage(qb) or 'none')

## Build the 5 role networks

In [ ]:
nets = build_all_networks()
for role, n in nets.items():
    print(f'{role:20s} nodes={sorted(n.model.nodes())}')

## Causal / evidential reasoning
Observing that a student answered a prerequisite skill correctly raises our belief in the skills that depend on it.

In [ ]:
fe = nets['Frontend Developer']
print('P(javascript mastered), no evidence     :', round(fe.query('javascript'), 3))
print('P(javascript mastered | html mastered)  :', round(fe.query('javascript', {'html': 1}), 3))
print('P(javascript mastered | html NOT mastered):', round(fe.query('javascript', {'html': 0}), 3))
print()
print('All beliefs after html=correct, css=wrong:')
print({k: round(v, 2) for k, v in fe.beliefs({'html': 1, 'css': 0}).items()})

## Independencies (Unit I)
`testing` and `performance` are both caused by `javascript` (a *common cause*): dependent when nothing is known, independent once `javascript` is observed. `check_independence` returns **True when d-connected (dependent)**.

In [ ]:
print('testing vs performance, nothing observed      -> dependent?', fe.check_independence('testing','performance'))
print('testing vs performance, given javascript     -> dependent?', fe.check_independence('testing','performance',['javascript']))
print('html vs performance, nothing observed         -> dependent?', fe.check_independence('html','performance'))
print('html vs performance, given css                -> dependent?', fe.check_independence('html','performance',['css']))
be = nets['Backend Developer']
print('backend: languages vs databases (unrelated roots) -> dependent?', be.check_independence('languages','databases'))

## Markov chain — interview state
The state after each technical answer depends only on the current state and whether the answer was right (Markov property).

In [ ]:
import numpy as np
np.random.seed(4)
mc = InterviewMarkovChain()
for correct in [True, True, False, False, True, True]:
    mc.step(correct)
print(' -> '.join(mc.history))

## Full engine demo (what the app runs)

In [ ]:
from interview_engine import InterviewSession
s = InterviewSession('Frontend Developer', qb, n_tech=5, n_hr=1, seed=7)
for i, q in enumerate(s.questions):
    ans = 'a long thoughtful answer ' * 12 if q['is_hr'] else q['expected_keywords'].replace(',', ' ')
    r = s.submit(i, ans, latency_seconds=10)
    print(('HR  ' if r['is_hr'] else r['skill'].ljust(12)), f"score={r['score']:.0%}  confidence={r['confidence']:.0%}  | {q['question'][:55]}")
rep = s.report()
print('\noverall confidence:', f"{rep['overall_confidence']:.0%}", '| weak skills:', rep['weak_skills'] or 'none')